# 🤖 Text-Based Autonomous Customer Support Agent
### LangGraph | LLM Classification | RAG | Escalation Ready

This notebook builds a **fully runnable autonomous customer support agent** using **LangGraph**.


In [ ]:
!pip install -U langgraph langchain langchain-community transformers sentence-transformers faiss-cpu pydantic


In [ ]:
from typing import TypedDict
from datetime import datetime
import uuid

from langgraph.graph import StateGraph, END
from langchain_community.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline


In [ ]:
class SupportState(TypedDict):
    customer_id: str
    ticket_id: str
    customer_message: str
    issue_category: str
    knowledge_context: str
    resolution: str
    status: str
    escalation_required: bool


In [ ]:
def today():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def generate_id():
    return str(uuid.uuid4())[:8]


In [ ]:
llm_pipeline = pipeline(
    'text-generation',
    model='google/flan-t5-base',
    max_new_tokens=64
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)


In [ ]:
support_docs = [
    'Order issues include delayed, missing, or failed orders.',
    'Payment issues involve failed transactions or double charges.',
    'Refunds are processed within 5-7 business days.',
    'Account issues include login failures and access problems.',
    'Technical issues relate to app crashes or website errors.'
]

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_texts(support_docs, embeddings)
retriever = vectorstore.as_retriever()


In [ ]:
MEMORY_DB = []


In [ ]:
def intake_node(state):
    state['ticket_id'] = generate_id()
    state['status'] = 'Received'
    return state


In [ ]:
def classify_issue(state):
    prompt = (
        'Classify the customer issue into one category: '
        'Order, Payment, Refund, Account, Technical.\n\n'
        f"Issue: {state['customer_message']}\n"
        'Respond with only the category name.'
    )

    response = llm.invoke(prompt).strip()
    valid = ['Order', 'Payment', 'Refund', 'Account', 'Technical']

    state['issue_category'] = response if response in valid else 'Technical'
    state['status'] = 'Classified'
    return state


In [ ]:
def rag_node(state):
    docs = retriever.invoke(state['customer_message'])
    state['knowledge_context'] = docs[0].page_content if docs else ''
    return state


In [ ]:
def decision_node(state):
    state['escalation_required'] = state['issue_category'] in ['Payment', 'Account']
    return state


In [ ]:
def response_node(state):
    prompt = f"""
Customer issue: {state['customer_message']}
Knowledge base: {state['knowledge_context']}

Provide a clear and helpful support response.
"""

    state['resolution'] = llm.invoke(prompt)
    state['status'] = 'Resolved'
    return state


In [ ]:
def memory_node(state):
    MEMORY_DB.append(state.copy())
    return state


In [ ]:
graph = StateGraph(SupportState)

graph.add_node('intake', intake_node)
graph.add_node('classify', classify_issue)
graph.add_node('rag', rag_node)
graph.add_node('decide', decision_node)
graph.add_node('respond', response_node)
graph.add_node('memory', memory_node)

graph.set_entry_point('intake')
graph.add_edge('intake', 'classify')
graph.add_edge('classify', 'rag')
graph.add_edge('rag', 'decide')
graph.add_edge('decide', 'respond')
graph.add_edge('respond', 'memory')
graph.add_edge('memory', END)

app = graph.compile()


In [ ]:
input_state = {
    'customer_id': 'CUST-101',
    'customer_message': 'My payment was deducted but order failed',
    'ticket_id': '',
    'issue_category': '',
    'knowledge_context': '',
    'resolution': '',
    'status': '',
    'escalation_required': False
}

result = app.invoke(input_state)
result


In [ ]:
MEMORY_DB
